In [1]:
import spacy
import os
#import pysbd
#from nltk.tokenize import sent_tokenize
from spacy.language import Language

path_to_scispacy_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\en_ner_bionlp13cg_md-0.5.4\\en_ner_bionlp13cg_md\\en_ner_bionlp13cg_md-0.5.4')
path_to_custom_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\custom_ner')

scispacy_model = spacy.load(path_to_custom_model, disable=['parser'])

sentencizer = scispacy_model.add_pipe('sentencizer', after='ner')
#seg = pysbd.Segmenter('en', False)

""" @Language.component('prevent_entity_splits')
def prevent_entity_splits(doc):
    # Iterate over all entities in the document
    for ent in doc.ents:
        # For each token inside the entity, set is_sent_start to False
        # This prevents it from starting a new sentence
        for token in ent[1:]:  # Start from the second token of the entity
            token.is_sent_start = False
    return doc

scispacy_model.add_pipe("prevent_entity_splits", after='ner') """

@Language.component('custom_sentencizer')
def custom_sentencizer(doc):
    for ent in doc.ents:
        for t in range(len(ent)-1):
            doc[ent.start+t+1].is_sent_start = False
    return doc

sentenciser = scispacy_model.add_pipe('custom_sentencizer', after='sentencizer')

print(scispacy_model.pipe_names)

c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'entity_ruler', 'ner', 'sentencizer', 'custom_sentencizer']


In [3]:
sentences = [
    'The enzymes in the food included lactase, glucosidase, kinase and amylase',
    'Bread consists of carbohydrates which break down into glucose. This can be done so by S. cerevisiae.',
    'L. casei play a decisive role in the maturation of pickles. Pickles do taste good.',
    'A microbe called Lpb. plantarum is short for Lactiplantibacillus plantarum. I hope there are 2 sentences.',
    'However, a fake one was invented called L. slksdus strain',
    'Various LAB strains, including Leuconostoc citreum 4452, Leuconostoc mesenteroides 2194, Weissella minor 4451, L. rhamnosus 1473, L. plantarum 4932, L. paracasei 4186, L. citreum 4452, P. acidilactici 3992, and their cocultures. This approach demonstrated significant potential for recovering additional quantities of valuable bioproducts. SSF of other agricultural residues, such as sugarcane bagasse, rice straw, wheat straw, rice bran, corn cobs, banana peels, and pomegranate peels, has also been explored for vanillin production.',
    'It has been shown that Sacc. cerevisiae could grow in milk and produce small amounts of ethanol, glycerol and lactic acid.'
]

#scispacy_model.remove_pipe('custom_sentencizer')

for sent in sentences:
    """ sent_split = sent_tokenize(sent)
    print(sent_split) """
    print(sent)
    sci = scispacy_model(sent)
    print(sci.ents)
    print([f"{s}" for s in sci.sents])
    print([(f"({s.ents})") for s in sci.sents])
    print()
    print()

The enzymes in the food included lactase, glucosidase, kinase and amylase
(lactase, glucosidase, kinase, amylase)
['The enzymes in the food included lactase, glucosidase, kinase and amylase']
['([lactase, glucosidase, kinase, amylase])']


Bread consists of carbohydrates which break down into glucose. This can be done so by S. cerevisiae.
(carbohydrates, glucose, S. cerevisiae)
['Bread consists of carbohydrates which break down into glucose.', 'This can be done so by S. cerevisiae.']
['([carbohydrates, glucose])', '([S. cerevisiae])']


L. casei play a decisive role in the maturation of pickles. Pickles do taste good.
(L. casei,)
['L. casei play a decisive role in the maturation of pickles.', 'Pickles do taste good.']
['([L. casei])', '([])']


A microbe called Lpb. plantarum is short for Lactiplantibacillus plantarum. I hope there are 2 sentences.
(Lpb. plantarum, Lactiplantibacillus plantarum)
['A microbe called Lpb. plantarum is short for Lactiplantibacillus plantarum.', 'I hope the